In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

In [13]:
pd.options.display.max_columns = 100
pd.options.display.max_colwidth = 200
pd.options.display.max_rows = 100

In [15]:
df_internal = pd.read_excel("cibil_score/Internal_Bank_Dataset.xlsx")

FileNotFoundError: [Errno 2] No such file or directory: 'cibil_score/Internal_Bank_Dataset.xlsx'

In [14]:
df_external = pd.read_excel("cibil_score/External_Cibil_Dataset.xlsx")

In [15]:
df_internal.columns

Index(['PROSPECTID', 'Total_TL', 'Tot_Closed_TL', 'Tot_Active_TL',
       'Total_TL_opened_L6M', 'Tot_TL_closed_L6M', 'pct_tl_open_L6M',
       'pct_tl_closed_L6M', 'pct_active_tl', 'pct_closed_tl',
       'Total_TL_opened_L12M', 'Tot_TL_closed_L12M', 'pct_tl_open_L12M',
       'pct_tl_closed_L12M', 'Tot_Missed_Pmnt', 'Auto_TL', 'CC_TL',
       'Consumer_TL', 'Gold_TL', 'Home_TL', 'PL_TL', 'Secured_TL',
       'Unsecured_TL', 'Other_TL', 'Age_Oldest_TL', 'Age_Newest_TL'],
      dtype='str')

In [16]:
df_external.columns

Index(['PROSPECTID', 'time_since_recent_payment',
       'time_since_first_deliquency', 'time_since_recent_deliquency',
       'num_times_delinquent', 'max_delinquency_level',
       'max_recent_level_of_deliq', 'num_deliq_6mts', 'num_deliq_12mts',
       'num_deliq_6_12mts', 'max_deliq_6mts', 'max_deliq_12mts',
       'num_times_30p_dpd', 'num_times_60p_dpd', 'num_std', 'num_std_6mts',
       'num_std_12mts', 'num_sub', 'num_sub_6mts', 'num_sub_12mts', 'num_dbt',
       'num_dbt_6mts', 'num_dbt_12mts', 'num_lss', 'num_lss_6mts',
       'num_lss_12mts', 'recent_level_of_deliq', 'tot_enq', 'CC_enq',
       'CC_enq_L6m', 'CC_enq_L12m', 'PL_enq', 'PL_enq_L6m', 'PL_enq_L12m',
       'time_since_recent_enq', 'enq_L12m', 'enq_L6m', 'enq_L3m',
       'MARITALSTATUS', 'EDUCATION', 'AGE', 'GENDER', 'NETMONTHLYINCOME',
       'Time_With_Curr_Empr', 'pct_of_active_TLs_ever',
       'pct_opened_TLs_L6m_of_L12m', 'pct_currentBal_all_TL', 'CC_utilization',
       'CC_Flag', 'PL_utilization', 'PL_Fla

In [17]:
 df = pd. merge ( df_internal, df_external, how ='inner', left_on = ['PROSPECTID'], right_on = ['PROSPECTID'] )

In [18]:
categorical_cols = df.select_dtypes(include=['object', 'string']).columns

print(categorical_cols)

Index(['MARITALSTATUS', 'EDUCATION', 'GENDER', 'last_prod_enq2',
       'first_prod_enq2', 'Approved_Flag'],
      dtype='str')


In [19]:
len(categorical_cols)

6

In [20]:
# Target
y = df['Credit_Score']

# Features
X = df.drop(['Credit_Score', 'Approved_Flag'], axis=1)

# Find categorical columns
categorical_cols = X.select_dtypes(include=['object', 'string']).columns

# One-hot encode
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Verify
print(X.select_dtypes(include=['object', 'string']).columns)

Index([], dtype='str')


In [21]:
results = []

In [22]:
features = X.columns.tolist()

for i in range(1, len(features) + 1):

    # Select first i features
    selected_features = features[:i]

    X_subset = X[selected_features]

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset,
        y,
        test_size=0.2,
        random_state=42
    )

    # Train model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # R²
    r2 = r2_score(y_test, y_pred)

    # Mean Squared Error (Loss)
    mse = mean_squared_error(y_test, y_pred)

    # Adjusted R²
    n = len(y_test)
    p = X_subset.shape[1]

    adjusted_r2 = 1 - ((1-r2)*(n-1))/(n-p-1)

    # Store results
    results.append([
        i,
        selected_features[-1],
        mse,
        r2,
        adjusted_r2
    ])

In [23]:
results_df = pd.DataFrame(
    results,
    columns=[
        "No_of_Features",
        "Last_Feature_Added",
        "Loss(MSE)",
        "R2",
        "Adjusted_R2"
    ]
)

results_df

,No_of_Features,Last_Feature_Added,Loss(MSE),R2,Adjusted_R2
0,1,PROSPECTID,418.476898,-0.000439,-0.000537
1,2,Total_TL,403.945599,0.034300,0.034112
2,3,Tot_Closed_TL,403.125582,0.036260,0.035979
3,4,Tot_Active_TL,403.125582,0.036260,0.035885
4,5,Total_TL_opened_L6M,378.869256,0.094249,0.093808
5,6,Tot_TL_closed_L6M,378.450545,0.095250,0.094721
6,7,pct_tl_open_L6M,372.818742,0.108714,0.108106
7,8,pct_tl_closed_L6M,371.593386,0.111643,0.110951
8,9,pct_active_tl,364.588712,0.128389,0.127625
9,10,pct_closed_tl,364.588712,0.128389,0.127540


In [24]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

print("R2 :", r2_score(y_test, y_pred))
print("MAE :", mean_absolute_error(y_test, y_pred))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred)))

R2 : 0.858062565181559
MAE : 6.176716581923089
RMSE : 7.705286805092864


In [25]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)

ridge.fit(X_train, y_train)

y_pred = ridge.predict(X_test)

In [26]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)

lasso.fit(X_train, y_train)

y_pred = lasso.predict(X_test)

C:\Users\Kamal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.719955e+06, tolerance: 1.728e+03
  model = cd_fast.enet_coordinate_descent(


In [27]:
from sklearn.linear_model import ElasticNet

elastic = ElasticNet(alpha=0.1,
                     l1_ratio=0.5)

elastic.fit(X_train, y_train)

y_pred = elastic.predict(X_test)

C:\Users\Kamal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.830128e+06, tolerance: 1.728e+03
  model = cd_fast.enet_coordinate_descent(


In [28]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# Create model
linear_model = LinearRegression()

# Train model
linear_model.fit(X_train, y_train)

# Prediction
y_pred_linear = linear_model.predict(X_test)

# Evaluation
r2 = r2_score(y_test, y_pred_linear)
mae = mean_absolute_error(y_test, y_pred_linear)
mse = mean_squared_error(y_test, y_pred_linear)
rmse = np.sqrt(mse)

print("----- Linear Regression -----")
print("R2 Score :", r2)
print("MAE :", mae)
print("MSE :", mse)
print("RMSE :", rmse)

----- Linear Regression -----
R2 Score : 0.858062565181559
MAE : 6.176716581923089
MSE : 59.371444748738185
RMSE : 7.705286805092864


In [29]:
n = X_test.shape[0]      # Number of observations
p = X_test.shape[1]      # Number of features

adj_r2 = 1 - ((1-r2)*(n-1)/(n-p-1))

print("Adjusted R2 :", adj_r2)

Adjusted R2 : 0.856694695320982


In [30]:
from sklearn.linear_model import Ridge

# Create Ridge model
ridge_model = Ridge(alpha=1.0)

# Train
ridge_model.fit(X_train, y_train)

# Prediction
y_pred_ridge = ridge_model.predict(X_test)

# Evaluation
ridge_r2 = r2_score(y_test, y_pred_ridge)
ridge_mae = mean_absolute_error(y_test, y_pred_ridge)
ridge_mse = mean_squared_error(y_test, y_pred_ridge)
ridge_rmse = np.sqrt(ridge_mse)

ridge_adj_r2 = 1 - ((1-ridge_r2)*(n-1)/(n-p-1))

print("\n----- Ridge Regression -----")
print("R2 Score :", ridge_r2)
print("Adjusted R2 :", ridge_adj_r2)
print("MAE :", ridge_mae)
print("MSE :", ridge_mse)
print("RMSE :", ridge_rmse)


----- Ridge Regression -----
R2 Score : 0.8584107066543876
Adjusted R2 : 0.8570461918792995
MAE : 6.167627735977977
MSE : 59.22581958476906
RMSE : 7.695831312130553


In [31]:
from sklearn.linear_model import Lasso

# Create Lasso model
lasso_model = Lasso(alpha=0.1)

# Train
lasso_model.fit(X_train, y_train)

# Prediction
y_pred_lasso = lasso_model.predict(X_test)

# Evaluation
lasso_r2 = r2_score(y_test, y_pred_lasso)
lasso_mae = mean_absolute_error(y_test, y_pred_lasso)
lasso_mse = mean_squared_error(y_test, y_pred_lasso)
lasso_rmse = np.sqrt(lasso_mse)

lasso_adj_r2 = 1 - ((1-lasso_r2)*(n-1)/(n-p-1))

print("\n----- Lasso Regression -----")
print("R2 Score :", lasso_r2)
print("Adjusted R2 :", lasso_adj_r2)
print("MAE :", lasso_mae)
print("MSE :", lasso_mse)
print("RMSE :", lasso_rmse)


----- Lasso Regression -----
R2 Score : 0.6041638463166537
Adjusted R2 : 0.6003491208705953
MAE : 9.464778022386765
MSE : 165.57551824172074
RMSE : 12.867615095336072


C:\Users\Kamal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.719955e+06, tolerance: 1.728e+03
  model = cd_fast.enet_coordinate_descent(


In [32]:
from sklearn.linear_model import ElasticNet

# Create Elastic Net model
elastic_model = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42)

# Train the model
elastic_model.fit(X_train, y_train)

# Predict
y_pred_elastic = elastic_model.predict(X_test)

# Evaluation
elastic_r2 = r2_score(y_test, y_pred_elastic)
elastic_mae = mean_absolute_error(y_test, y_pred_elastic)
elastic_mse = mean_squared_error(y_test, y_pred_elastic)
elastic_rmse = np.sqrt(elastic_mse)

# Adjusted R2
n = X_test.shape[0]
p = X_test.shape[1]

elastic_adj_r2 = 1 - ((1 - elastic_r2) * (n - 1) / (n - p - 1))

print("----- Elastic Net Regression -----")
print("R2 Score :", elastic_r2)
print("Adjusted R2 :", elastic_adj_r2)
print("MAE :", elastic_mae)
print("MSE :", elastic_mse)
print("RMSE :", elastic_rmse)

----- Elastic Net Regression -----
R2 Score : 0.5972081290503191
Adjusted R2 : 0.5933263704356009
MAE : 9.49958363243891
MSE : 168.48504654124423
RMSE : 12.980178987257618


C:\Users\Kamal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.830128e+06, tolerance: 1.728e+03
  model = cd_fast.enet_coordinate_descent(


In [33]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Lasso Regression",
        "Elastic Net Regression"
    ],
    "R2 Score": [
        r2,
        ridge_r2,
        lasso_r2,
        elastic_r2
    ],
    "Adjusted R2": [
        adj_r2,
        ridge_adj_r2,
        lasso_adj_r2,
        elastic_adj_r2
    ],
    "MAE": [
        mae,
        ridge_mae,
        lasso_mae,
        elastic_mae
    ],
    "MSE": [
        mse,
        ridge_mse,
        lasso_mse,
        elastic_mse
    ],
    "RMSE": [
        rmse,
        ridge_rmse,
        lasso_rmse,
        elastic_rmse
    ]
})

print(comparison)

                    Model  R2 Score  Adjusted R2       MAE         MSE  \
0       Linear Regression  0.858063     0.856695  6.176717   59.371445   
1        Ridge Regression  0.858411     0.857046  6.167628   59.225820   
2        Lasso Regression  0.604164     0.600349  9.464778  165.575518   
3  Elastic Net Regression  0.597208     0.593326  9.499584  168.485047   

        RMSE  
0   7.705287  
1   7.695831  
2  12.867615  
3  12.980179  


Ridge Regression performed the best.
Highest R² Score (0.8584).
Highest Adjusted R² (0.8570).
Lowest MAE (6.1676).
Lowest RMSE (7.6958).

Ridge Regression produced the best predictive performance among all four models. The L2 regularization reduced coefficient magnitudes without eliminating important features, which improved generalization and slightly reduced prediction error compared to ordinary Linear Regression.

Linear Regression also performed very well.
R² = 0.8581
RMSE = 7.7053

The performance is extremely close to Ridge.

Difference in R²:

0.858411 − 0.858063 = 0.000348

This improvement is very small (about 0.035 percentage points), indicating that the dataset does not suffer from severe multicollinearity or overfitting.

Lasso Regression performed much worse.

Compared to Linear Regression:

R² decreased from 0.858 to 0.604.
RMSE increased from 7.70 to 12.87.

Reason:
Lasso (L1 regularization) shrinks some coefficients exactly to zero. In  CIBIL dataset, many predictors appear to contribute useful information. By removing some of them, Lasso underfits the data, reducing predictive accuracy.

Elastic Net also performed poorly.

Elastic Net combines Ridge and Lasso.

However:

R² = 0.597
RMSE = 12.98

Its performance is very similar to Lasso, suggesting that the L1 component dominates with your chosen hyperparameters (alpha and l1_ratio) and causes underfitting.

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
logistic_model = LogisticRegression(
    solver='saga',
    max_iter=1000,
    random_state=42
)